# 🏃 Notebook 3: Threaded Motion Control & Safe Jogging

This notebook verifies physical or simulated hardware motion control by sending movement commands to axes that have completed the Servo On sequence. 

You will practice:
1. **Constant Velocity Control**: Moving the motor at a designated speed and stopping it.
2. **Hold-to-move Jogging**: A safety-critical jogging mechanism that automatically halts the motor if the control cycle is interrupted.
3. **Interactive HMI Control**: Operating the motor via user-friendly dashboard buttons and testing an **Emergency Stop (E-Stop)**.

*Prerequisite: You must execute `01_wmx_system_startup.ipynb` first to ensure that the designated axes are in the Servo On state.*

In [3]:
import rclpy
from wmx_r2_message.msg import AxisVelocity
from wmx_r2_message.srv import SetAxis
import ipywidgets as widgets
from IPython.display import display
import threading
import time
from wmx_utils import WmxClient

if not rclpy.ok():
    rclpy.init()

class WmxHmiController(WmxClient):
    def __init__(self):
        super().__init__(node_name='wmx_hmi_controller')
        
        self.vel_pub = self.create_publisher(AxisVelocity, '/wmx/axis/velocity', 10)
        self.jog_pub = self.create_publisher(AxisVelocity, '/wmx/axis/jog', 10)
        self.stop_cli = self.create_client(SetAxis, '/wmx/axis/stop')
        
        self.direction = 0
        self.is_running = False
        self.position = 0.0


        self.lbl_status = widgets.Label(value="Status: IDLE (Stopped)")
        self.lbl_pos = widgets.HTML(value="<h3>Current Position: 0.0 deg</h3>")

    def _loop(self):
        target_speed = 1000.0 if self.direction > 0 else -10000.0
        speeds = [target_speed] * len(self.axis_list)
        accs = [100.0] * len(self.axis_list)
        decs = [100.0] * len(self.axis_list)
        
        # Send initial continuous motion command to WMX core
        self.publish_vel(self.axis_list, speeds, accs, decs)
        
        while self.is_running:
            # 50ms interval loop (20 FPS) for real-time browser feedback simulation
            self.position += self.direction * 1.5
            self.lbl_pos.value = f"<h3>Current Position: <span style='color: #007acc;'>{self.position:.1f} deg</span></h3>"
            time.sleep(0.05)
            
        # Decelerate all controlled axes to a safe stop when STOP is clicked
        zero_speeds = [0.0] * len(self.axis_list)
        self.publish_vel(self.axis_list, zero_speeds, accs, decs)

    def publish_vel(self, index, velocity, acc, dec):
        msg = AxisVelocity()
        msg.index = index
        msg.velocity = velocity
        msg.acc = acc
        msg.dec = dec
        self.vel_pub.publish(msg)

    def start_move(self, direction):
        if not self.is_running:
            self.direction = direction
            self.is_running = True
            dir_text = "Rotating Positive (CW) 🔄" if direction > 0 else "Rotating Negative (CCW) 🔄"
            self.lbl_status.value = f"Status: {dir_text}"
            threading.Thread(target=self._loop, daemon=True).start()

    def stop_move(self):
        self.is_running = False
        self.direction = 0
        self.lbl_status.value = "Status: IDLE (Stopped)"

    def call_stop(self):
        while not self.stop_cli.wait_for_service(timeout_sec=1.0):
            pass
        req = SetAxis.Request()
        req.index = self.axis_list
        # Automatically generate zero matching data arrays for all target axes
        req.data = [0] * len(self.axis_list)
        
        future = self.stop_cli.call_async(req)
        rclpy.spin_until_future_complete(self, future)
        return future.result()

# Instantiate the controller
hmi = WmxHmiController()
print(f"✅ HMI Controller Ready. Controlling Axes: {hmi.axis_list}")

✅ HMI Controller Ready. Controlling Axes: [0]


[WARN] [1787616425.283825422] [rcl.logging_rosout]: Publisher already registered for provided node name. If this is due to multiple nodes with the same name then all logs for that logger name will go out over the existing publisher. As soon as any node with that name is destructed it will unregister the publisher, preventing any further logs for that name from being published on the rosout topic.


### 1. Interactive Dashboard (HMI) Control
* This control dashboard uses a multi-threaded Python approach combined with `ipywidgets`.
* Click **[▶ Positive (CW)]** or **[◀ Negative (CCW)]** to spin the actual motor. The thread updates the browser with simulated coordinate feedback in real-time.
* Click **[■ STOP]** to safely bring the physical motor to a halt.

In [4]:
# Instantiate control dashboard buttons
btn_cw = widgets.Button(description="▶ Positive (CW)", button_style="success", layout=widgets.Layout(width="140px", height="40px"))
btn_stop = widgets.Button(description="■ STOP", button_style="danger", layout=widgets.Layout(width="90px", height="40px"))
btn_ccw = widgets.Button(description="◀ Negative (CCW)", button_style="warning", layout=widgets.Layout(width="140px", height="40px"))

# Bind buttons directly to controller state machine functions
btn_cw.on_click(lambda _: hmi.start_move(1))
btn_ccw.on_click(lambda _: hmi.start_move(-1))
btn_stop.on_click(lambda _: hmi.stop_move())

# Arrange the GUI layout nicely and display
controls = widgets.HBox([btn_ccw, btn_stop, btn_cw], layout=widgets.Layout(justify_content="center", margin="10px 0"))
display_box = widgets.VBox([hmi.lbl_status, hmi.lbl_pos], layout=widgets.Layout(align_items="center"))
panel = widgets.VBox([controls, display_box], layout=widgets.Layout(border="1px solid #ccc", padding="15px", width="420px", align_items="center"))

print("📬 Operate the motor and monitor the live coordinate stream below:")
display(panel)


📬 Operate the motor and monitor the live coordinate stream below:


### 2. Safety Watchdog - Hold-to-move Jogging via Jupyter Terminal

* **The Jupyter Limitation**: In a standard browser-based Jupyter code cell, capturing real-time keyboard press-and-hold (`keydown`) and release (`keyup`) events with the low-latency precision required for industrial motors is not natively supported due to browser event-loop queuing.
* **The Real-world Solution**: Jupyter Notebook includes a built-in Ubuntu **Terminal**. We can utilize this to run the native, interactive keyboard jogging node (`jog_keyboard_node`) built into your `wmx-r2` repository!
* **Keyboard Mapping**:
  * Press and hold **`d`**: Jog Axis 0 in the **positive** direction.
  * Press and hold **`a`**: Jog Axis 0 in the **negative** direction.
  * Press **`q`**: Quit the jog node safely.
* **How It Works**: The terminal node detects the operating system's keyboard auto-repeat rate. While you hold down the key, auto-repeat characters keep streaming to the node, instructing the motor to spin. The instant you release the key, the character stream stops. The WMX3 motion engine's watchdog immediately senses this interruption and decelerates the motor to a safe stop.
[Cell 6] - Code Cell (Jupyter Terminal Run Guide & Command Generator)

In [11]:
# Run this cell to print setup instructions and generate the exact command 
# to copy-paste into your Jupyter Terminal.

import os

# Resolve the target axis to log in the command dynamically
target_axis_str = ",".join(map(str, hmi.axis_list))

print("=================================================================")
print("🏃 STEP-BY-STEP REAL KEYBOARD JOGGING GUIDE")
print("=================================================================")
print("1. Open the Jupyter Terminal:")
print("   👉 Click the Jupyter logo -> Select 'New' at the top-right -> Click 'Terminal'.")
print("\n2. Source your ROS 2 and workspace environment in the terminal:")
print("   $ source /opt/ros/jazzy/setup.bash")
print("   $ source ~/workspaces/movensys_ws/install/setup.bash")
print("\n3. Copy and run this command to start the interactive jogger:")
# =========================================================================
# 📍 MULTI-AXIS JOGGING CONFIGURATION:
# The command below dynamically matches the axis list you defined in Cell 2.
# =========================================================================
print(f"   $ ros2 run wmx_r2_package jog_keyboard_node --ros-args -p axis:=\"{target_axis_str}\" -p velocity:=5000.0 -p acc:=100.0 -p dec:=100.0")
print("\n4. Test the Safety Watchdog:")
print("   - Click inside the terminal window to focus.")
print("   - Press and HOLD the 'd' key. The motor will spin forward.")
print("   - RELEASE the 'd' key. Watch the motor immediately decelerate to a stop!")
print("   - Press and HOLD 'a' to spin backward, and release to stop.")
print("\n5. Exit the jogger safely by pressing 'q' inside the terminal.")
print("=================================================================")

🏃 STEP-BY-STEP REAL KEYBOARD JOGGING GUIDE
1. Open the Jupyter Terminal:
   👉 Click the Jupyter logo -> Select 'New' at the top-right -> Click 'Terminal'.

2. Source your ROS 2 and workspace environment in the terminal:
   $ source /opt/ros/jazzy/setup.bash
   $ source ~/workspaces/movensys_ws/install/setup.bash

3. Copy and run this command to start the interactive jogger:
   $ ros2 run wmx_r2_package jog_keyboard_node --ros-args -p axis:="1" -p velocity:=5000.0 -p acc:=100.0 -p dec:=100.0

4. Test the Safety Watchdog:
   - Click inside the terminal window to focus.
   - Press and HOLD the 'd' key. The motor will spin forward.
   - RELEASE the 'd' key. Watch the motor immediately decelerate to a stop!
   - Press and HOLD 'a' to spin backward, and release to stop.

5. Exit the jogger safely by pressing 'q' inside the terminal.
